# Generating Ground Truth Data

To evaluate search, we need a dataset of queries where we know which
document is the correct answer. This is called ground truth (or gold
standard) data.

For each query in our ground truth dataset, we know which document in
the knowledge base is relevant. When we run a search, we check whether
the results include the correct document.

There are several ways to get ground truth data:

- Human annotators look at documents and write queries (best quality, expensive)
- Collect real user queries and label them (requires a running system)
- Generate synthetic data with an LLM (what we'll do)

We don't have a production system yet, so we'll use an LLM to generate
questions. For each FAQ document, we ask the LLM to create 5 questions
that this document would answer. Then we know that for each generated
question, the source document is the correct answer.

## Loading the documents

We'll use helper files from module 01 and this module.

If you don't have them in your notebook directory, download them:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget ${PREFIX}/cohorts/2026/01-agentic-rag/code/ingest.py
wget ${PREFIX}/cohorts/2026/01-agentic-rag/code/rag_helper.py
wget ${PREFIX}/cohorts/2026/04-evaluation/code/evaluation_utils.py
```

In [33]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/04-evaluation/code/evaluation_utils.py

--2026-09-22 15:09:28--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3073 (3.0K) [text/plain]
Saving to: ‘evaluation_utils.py.2’

evaluation_utils.py 100%[===================>]   3.00K  --.-KB/s    in 0s      

2026-09-22 15:09:28 (41.8 MB/s) - ‘evaluation_utils.py.2’ saved [3073/3073]



Then load the FAQ data:

In [34]:
from ingest import load_faq_data
documents = load_faq_data()

We'll generate questions only for the LLM Zoomcamp FAQ. The full FAQ
dataset contains documents from multiple courses. Generating five
questions for every document would take longer and cost more.

In [35]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

153

We'll use these documents from now on so let's name them as `documents`

In [36]:
documents = documents_llm

Each document already has an `id` field:

In [37]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [38]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

The ID becomes the label in our ground truth dataset. We generate
questions from a document, so we know that this document holds the
answer. Later, search evaluation checks whether search brings back the
document with this ID.

This is why every record needs a stable ID. If you can't uniquely
identify a document, you can't tell whether search retrieved the right
one. When you build your own evaluation set, assign an ID to each record
in your knowledge base first.

## Generating questions with structured output

We use an LLM to generate questions for each document.

This is the first time we're using structured output in the course.
With structured output, we ask the LLM to return data in a specific
format instead of free-form text. For example, instead of getting a
paragraph that contains questions, we can ask for a Python object with
a `questions` field.

This is useful when code will process the output. The model returns the
same structure every time. We can access the generated questions
directly instead of parsing text manually.


We want the output as a list of strings, so we define that structure
with a Pydantic model:

In [39]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

The instructions for the LLM:

In [40]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

We ask the LLM to use different wording from the original document.
This makes the evaluation more realistic - real users won't phrase
their questions the same way as the FAQ.


Call the LLM for one document:

In [41]:
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.responses import ResponseInputItemParam

load_dotenv()
openai_client = OpenAI()

Prepare the document as JSON:

In [42]:
import json

user_prompt = json.dumps(doc)

In [43]:
print(user_prompt)

{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."}


Create the messages:

In [44]:
messages: list[ResponseInputItemParam] = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [45]:
messages

[{'role': 'developer',
  'content': "You emulate a student who's taking our course.\nFormulate 5 questions this student might ask based on a FAQ record. The record\nshould contain the answer to the questions, and the questions should be complete and not too short.\nIf possible, use as fewer words as possible from the record.\n\nThe output should resemble how people ask questions\non the internet. Not too formal, not too short, not too long."},
 {'role': 'user',
  'content': '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'}]

Until now we called `responses.create` and read `response.output_text`.
For structured output we switch to `responses.parse` and pass
`text_format=Questions`, which tells the API to return our class instead
of free text.

Call the model:

In [46]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [47]:
response

ParsedResponse[TypeVar](id='resp_052e1ba21fb0938f006ab29a2ac5c887d2bde6a8ab0ab41f11', created_at=1790089770.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ParsedResponseOutputMessage[TypeVar](id='msg_052e1ba21fb0938f006ab29a2b636887d281fbffe6ddf39cc8', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"questions":["I just found out about this course — is it still okay to join now?","If I join late, can I still get a certificate or did I miss that part?","Do I need to finish and submit a project before submissions close to get certified?","What’s the deadline for the project if I want the course certificate?","Can someone who discovers the course after it started still participate normally?"]}', type='output_text', logprobs=[], parsed=Questions(questions=['I just found out about this course — is it still okay to join now?', 'If I join late, can I still get a certificate or did I miss tha

The parsed object is available in `response.output_parsed`:

In [48]:
result = response.output_parsed
result

Questions(questions=['I just found out about this course — is it still okay to join now?', 'If I join late, can I still get a certificate or did I miss that part?', 'Do I need to finish and submit a project before submissions close to get certified?', 'What’s the deadline for the project if I want the course certificate?', 'Can someone who discovers the course after it started still participate normally?'])

We can access the list directly:

In [49]:
if result is not None:
    print(result.questions)

['I just found out about this course — is it still okay to join now?', 'If I join late, can I still get a certificate or did I miss that part?', 'Do I need to finish and submit a project before submissions close to get certified?', 'What’s the deadline for the project if I want the course certificate?', 'Can someone who discovers the course after it started still participate normally?']


You should see 5 questions that relate to the first FAQ document.

## Reusable utilities

We'll need this pattern again in other evaluation sections today, so
we put it in a reusable helper.

It contains helper functions we'll reuse in this module:

- `llm_structured`: calls the OpenAI API with structured output
- `llm_structured_retry`: retries structured-output calls when a
  request fails
- `calc_price`: calculates the price from token usage
- `calc_total_price`: calculates the total price from multiple usage
  objects
- `map_progress`: runs work in parallel and tracks progress. We'll use it
  in the next lesson.

Import the structured-output helper:

In [50]:
from evaluation_utils import llm_structured

Use it on the same document:

In [51]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

if result is not None:
    print(result.questions)

['Can I still join the course if I found it late?', 'If I start now, is there still a way to get the certificate?', 'What do I need to do to be eligible for the certificate after joining late?', 'Is late enrollment allowed, or is the course already closed?', 'Can I participate now and still get credited if I finish the project on time?']


## Tracking cost

The response also contains token usage:

In [52]:
assert usage is not None
usage.input_tokens, usage.output_tokens

(207, 87)

As in the agents module, we calculate the price from `response.usage`.

Import the price helper:

In [53]:
from evaluation_utils import calc_price

Calculate the cost of this call:

In [54]:
cost = calc_price(usage)
cost

{'input_cost': 0.00015525,
 'output_cost': 0.00039150000000000003,
 'total_cost': 0.00054675}

Now convert these questions into ground truth records:

In [55]:
records = []

assert result is not None
for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'If I start now, is there still a way to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate after joining late?',
  'document': '74eb249bbf'},
 {'question': 'Is late enrollment allowed, or is the course already closed?',
  'document': '74eb249bbf'},
 {'question': 'Can I participate now and still get credited if I finish the project on time?',
  'document': '74eb249bbf'}]

Each record has two fields:

- `question`: the question generated by the LLM
- `document`: the ID of the FAQ document that should answer the question

The `document` field connects the generated question to the document
that contains the answer. Later, when we evaluate search, we'll ask the
search engine the generated question. Then we'll check if it retrieves
the document with this ID.

We now know how to generate and store questions for one document. In
the next lesson, we'll run this for all LLM Zoomcamp FAQ documents and
save the full ground truth dataset.

# Generating Ground Truth for All Documents

In the previous lesson, we generated questions for one document and
converted them into ground truth records.

We want to do the same thing for every document in the FAQ dataset.
For each document, we generate questions and save them as ground truth
records.

For this part, we'll use `tqdm` for progress bars and `pandas` for
saving the final CSV.

If you don't have them installed yet, add them first:

```bash
uv add tqdm pandas
```

The processing function takes one document and turns it into ground
truth records.

For each document, we:

- convert the document to JSON so we can send it to the LLM
- ask the LLM to return a `Questions` object
- create one ground truth record for each generated question

Each record contains the generated question and the ID of the document
that should answer the question.

When we send many requests, one of them might fail. We don't want the
entire batch to fail because of one temporary error.

Import the retry helper from `evaluation_utils.py`:


In [56]:
import pandas as pd
from evaluation_utils import llm_structured_retry

In [57]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,"If I start now, is there still a way to get th...",74eb249bbf
2,What do I need to do to be eligible for the ce...,74eb249bbf
3,"Is late enrollment allowed, or is the course a...",74eb249bbf
4,Can I participate now and still get credited i...,74eb249bbf


`llm_structured` makes one structured-output call. `llm_structured_retry`
wraps the same call in a retry loop. If one request fails because of a
temporary API or network issue, it waits briefly and tries again.

Use it in the processing function:

In [58]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    ) # type: ignore

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [60]:
generate_ground_truth(doc)

([{'question': 'How do I find my own entry on the leaderboard if I can’t see my name right away?',
   'document': 'c2903069a0'},
  {'question': 'Why am I showing up with a random name on the leaderboard, and can I change it?',
   'document': 'c2903069a0'},
  {'question': 'Where can I check what my displayed name is in the course profile?',
   'document': 'c2903069a0'},
  {'question': 'What should I put in the profile fields if I want a different nickname on the leaderboard?',
   'document': 'c2903069a0'},
  {'question': 'How do I make sure my real name appears on the certificate instead of the default random one?',
   'document': 'c2903069a0'}],
 ResponseUsage(input_tokens=381, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=106, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=487))

Try it for the first 5 documents.

Import `tqdm` and run the loop:

In [59]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

This works, but it runs one LLM call after another. Running it for all
documents this way would take too long.

## Parallel processing

Running the calls one after another wastes most of the time waiting on
the network. Each request just sits there until OpenAI responds, so we
can fire several at once and wait on them together. We process the
documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the
provider's rate limits. Five or six workers is a safe default here.

![Splitting the documents into parts to process in parallel](images/03-ground-truth-batch-03-parallel-split-whiteboard-imagegen.png)

Import `ThreadPoolExecutor`:

In [61]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

This submits one job per document, updates the progress bar when a job
finishes, and collects the results. If you want a more detailed
explanation of `ThreadPoolExecutor` and futures, ask ChatGPT to walk
through this helper line by line.

Then replace the loop with the parallel version:

In [62]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/153 [00:00<?, ?it/s]

In [73]:
results[1]
len(results)

153

In [66]:
ground_truth[1]

{'question': 'If I start late, can I still take part in the course and work through the materials?',
 'document': '74eb249bbf'}

`generate_ground_truth` returns two things for each document: the
generated records and the token usage.

Split those into separate lists:

In [63]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

765

With 5 questions per document, you should get roughly 5x the number of
documents.

Calculate the total cost:

In [64]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.12385575000000003

We'll calculate total cost several times in this module, so the utility
file has a helper for it:

In [68]:
from evaluation_utils import calc_total_price

calc_total_price(usages)


0.12385575000000003

Create a dataframe so we can look at the records as a table and save
them as a CSV file.

Create the dataframe:

In [69]:
df_ground_truth = pd.DataFrame(ground_truth)

In [70]:
df_ground_truth

,question,document
0,I just found this course — is it still okay to...,74eb249bbf
1,"If I start late, can I still take part in the ...",74eb249bbf
2,Am I allowed to join after the course has alre...,74eb249bbf
3,Will I still be able to get a certificate if I...,74eb249bbf
4,What’s the deadline if I want to earn the cert...,74eb249bbf
...,...,...
760,How can I switch the dlt workshop agent from O...,54b98d3581
761,What do I need to put in `homework/agent.py` i...,54b98d3581
762,Do I still need an OpenAI key for the workshop...,54b98d3581
763,Which model string should I use with Groq in t...,54b98d3581


Because we generated the questions from specific documents, we know
which document is correct for each question. We now have the ground
truth we need for evaluation.

Save it for later use:

In [72]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

The run used 153 LLM Zoomcamp documents and produced 765 questions.

The FAQ data can change over time. If you run the notebook later, you
may see different documents and generated questions. Token usage, cost,
and search evaluation results may also change.

The total cost was $0.123856, about 12 cents.

If you don't want to generate the questions yourself, download the file
we prepared:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget -O data/ground_truth-new.csv ${PREFIX}/cohorts/2026/04-evaluation/data/ground_truth-new.csv
```

Now we have questions with known correct documents. In the next lesson,
we'll run search for these questions and check whether the correct
documents appear in the results.